# 05. 모형 비교 (Naive → N → A / B)

**이 노트북이 하는 일**

팀원의 단순 SARIMA(Naive)를 우리 데이터로 재현하고, 우리 모형 **N**(날씨 없음), **A**(특보만), **B_lag**(ASOS 전날까지), **B**(ASOS 당일 포함)와 **2024년**에서 나란히 비교합니다 (RQ3). 두 가지 방식으로 평가합니다.

- **(가) 하루 앞 롤링**: 학습(2014~2023) 계수를 고정하고, 2024년 각 날을 **전날까지의 실제값**으로 예측합니다.
- **(나) 365일 한 번에**: **2023-12-31까지의 자료만으로 2024년 366일 전체**를 한 번에 예측합니다 (팀원 노트북의 방식).

**모형**

| 모형 | 내용 |
|---|---|
| **Naive** | 팀원 `Naive_SARIMA.ipynb` 재현: 이용량(원 단위, log 아님), 코로나(2020-01-01~2022-12-31)만 NaN, 외생변수 없음, 역별 차수(`NAIVE_ORDERS`), `enforce_*` 끔, `maxiter=100` |
| **N** | 우리 기준 모형: log 이용량, `MASK_COLS` 가림, 계단 더미 + `summer_vacation`, (1,0,1)(1,1,1,7) |
| **A** | N + 특보 4개와 lag1~3 |
| **B_lag** | N + ASOS 날씨의 lag1~3만 (당일을 뺀 것: 예측 시점에 아는 정보) |
| **B** | N + ASOS 날씨 당일과 lag1~3 (사후 상한선) |

**평가 규칙**

- **평가일은 우리 마스크(`MASK_COLS`)로 가린 날을 뺀 날로 통일**합니다 (343일, 시청 339일). Naive도 같은 날로만 점수를 냅니다. 팀원 방식(366일 전체)의 검증 MAE는 재현 확인용으로 따로 계산합니다.
- **(나) 365일 한 번에에서 A, B_lag, B는 외생변수(특보·날씨)를 2024년 실제값으로 줍니다.** 2023-12-31 시점에는 알 수 없는 값이라 **사후 평가(상한선)**입니다. N과 Naive는 달력 더미 외에 외생변수가 없습니다.
- 지표는 원 단위 MAE, RMSE, MAPE입니다. **2025년 행은 읽자마자 버리며 쓰지 않습니다.**

**입력 파일**

- `data/processed/model_input_9stations.csv`, 설정(`NAIVE_ORDERS`, `NAIVE_COVID`, `NAIVE_VALID_MAE`)은 `00_settings`

**출력 파일**

- `results/model_compare_by_station.csv` (역 × 모형 × 방식), `model_compare_by_type.csv` (유형 평균), `model_compare_steps.csv` (Naive → N → A/B 단계별 개선), `model_compare_repro.csv` (팀원 결과 재현 확인), `model_compare_pred2024.csv` (일별 예측), `model_compare_fits.csv` (적합 수렴 여부)


**설정 불러오기**: Colab이면 저장소를 clone해 경로를 맞추고, `00_settings.ipynb`의 설정을 불러옵니다 (로컬이면 clone은 건너뜀).

In [1]:
# Colab에서 실행하면 저장소를 clone하고 notebooks/로 이동합니다 (로컬이면 건너뜁니다).
import os, sys, subprocess
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/TimeSeries'):
        subprocess.run(['git', 'clone', 'https://github.com/Eui-seung/TimeSeries.git', '/content/TimeSeries'], check=True)
    os.chdir('/content/TimeSeries/notebooks')

# 설정(경로, 역, 열 이름, 기간, 차수 등)을 불러옵니다.
%run ./00_settings.ipynb

필요한 패키지를 불러옵니다. 역별 계산은 `joblib`으로 병렬 처리합니다.

In [2]:
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

**입력 파일 경로**: 모델 입력 표.

In [3]:
DATA = PROC / 'model_input_9stations.csv'

아래 함수들은 `03_sarimax`와 **같은 코드**입니다 (우리 모형 N / A / B_lag / B의 입력 만들기, 적합, 하루 앞 롤링 검증). 03번 결과와 같은 값이 나오는지 아래에서 확인합니다.

**`load`**: `model_input_9stations.csv`를 읽고 2025(test) 행은 바로 버립니다.

In [4]:
def load():
    d = pd.read_csv(DATA, parse_dates=['date'])
    return d[d.date <= VALID_END]                              # 2025(test)는 여기서 버림

**`prepare`**: 한 역의 y, X(2014~2024 전체)와 유형. 학습·검증 분리는 호출하는 쪽에서 한다.

In [5]:
def prepare(d, station, season=False, model='B'):
    """한 역의 y, X(2014~2024 전체)와 유형. 학습·검증 분리는 호출하는 쪽에서 한다.
    model: 'B' = ASOS 실측만, 'A' = 특보만, 'C' = 특보 + ASOS, 'N' = 날씨 없음, 'B_lag'/'C_lag' = B/C에서 당일 ASOS를 뺀 버전"""
    s = d[d.station == station].set_index('date').sort_index().asfreq('D')
    assert s.index.min() == START and s.index.max() == VALID_END and s.station.notna().all(), '날짜 빈틈'
    y = np.log(s.total)
    y[s[MASK_COLS].sum(axis=1) > 0] = np.nan                 # 가리기: 날짜는 유지, y만 NaN
    blocks = []
    if model in ('B', 'C', 'B_lag', 'C_lag'):
        w = pd.DataFrame({
            'heat_excess': (s.feels_like_max - HEAT_BASE).clip(lower=0).fillna(0),
            'cold_excess': (COLD_BASE - s.tmin).clip(lower=0).fillna(0),
            'rain': np.log1p(s.rain_mm),
            'snow': s.snow_new_cm,
        }).fillna(0)
        blocks += ([] if model.endswith('_lag') else [w]) + [pd.DataFrame({f'{c}_lag{k}': w[c].shift(k).fillna(0) for c in WEATHER for k in LAGS})]
    if model in ('A', 'C', 'C_lag'):
        a = s[ALERTS].fillna(0)                                 # NaN은 2014-01-01~07뿐(y도 가려짐)
        blocks += [a, pd.DataFrame({f'{c}_lag{k}': a[c].shift(k).fillna(0) for c in ALERTS for k in LAGS})]
    steps = list(STEP_COLS) + (list(SEASON_COLS) if season else []) + ([STATION_STEP[station]] if station in STATION_STEP else [])
    X = pd.concat([s[steps]] + blocks, axis=1).astype(float)
    return y, X, s.group.iloc[0], s.total

**`_fit`**: SARIMAX를 한 번 적합합니다(경고를 모아 둠).

In [6]:
def _fit(model, start=None, maxiter=None, method=None, **kw):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        res = model.fit(start_params=start, disp=False, method=method or FIT_METHOD, maxiter=maxiter or FIT_MAXITER, **kw)
    return res, sorted({f'{w.category.__name__}: {str(w.message)[:110]}' for w in caught})

**`_converged`**: 적합이 수렴했는지 확인합니다.

In [7]:
def _converged(res):
    return bool(res.mle_retvals.get('converged'))

**`alt_start`**: 다른 초기값: 7일 차분한 y를 7일 차분한 X에 회귀한 계수 + 고정한 ARMA 초기값

In [8]:
def alt_start(model, ytr, Xtr):
    """다른 초기값: 7일 차분한 y를 7일 차분한 X에 회귀한 계수 + 고정한 ARMA 초기값"""
    dy, dX = ytr - ytr.shift(7), Xtr - Xtr.shift(7)
    ok = dy.notna() & dX.notna().all(axis=1)
    beta = np.linalg.lstsq(dX[ok].values, dy[ok].values, rcond=None)[0]
    sig2 = float(np.var(dy[ok].values - dX[ok].values @ beta))
    cols, out = list(Xtr.columns), []
    for n in model.param_names:
        if n in cols: out.append(beta[cols.index(n)])
        elif n.startswith('ar.S'): out.append(0.1)
        elif n.startswith('ma.S'): out.append(-0.8)
        elif n == 'ar.L1': out.append(0.5)
        elif n.startswith(('ar.', 'ma.')): out.append(0.0)
        else: out.append(sig2)
    return np.array(out)

**`fit_station`**: 2014~2023으로 적합. 반환: 결과, 경고 목록, 뺀 열, 사용한 X, 적합 방법 메모.

In [9]:
def fit_station(y, X, order, seasonal_order, retry=False, hint=None):
    """2014~2023으로 적합. 반환: 결과, 경고 목록, 뺀 열, 사용한 X, 적합 방법 메모.
    retry=True면 기본 설정(lbfgs, maxiter=1000)이 수렴하지 않을 때 순서대로 다시 시도:
    (1) 기본 결과의 계수에서 이어서 maxiter=5000, (2) 다른 초기값에서 maxiter=5000. 첫 수렴 결과를 쓰고, 없으면 loglike가 가장 큰 것을 씀.
    hint(다른 모형의 계수 {이름: 값})를 주면 그 계수를 초기값으로 lbfgs(maxiter=5000)를 먼저 돌리고, 수렴하지 않으면
    그 초기값에서 method='nm'을 먼저 돌린 뒤 그 결과로 lbfgs(maxiter=5000)를 다시 돌린다(없는 변수는 0). 그래도 안 되면 미수렴 표시"""
    ytr, Xtr = y[:TRAIN_END], X[:TRAIN_END]
    dropped = [c for c in X.columns if (Xtr[c] == 0).all()]
    X = X.drop(columns=dropped); Xtr = Xtr.drop(columns=dropped)
    model = SARIMAX(ytr, exog=Xtr, order=order, seasonal_order=seasonal_order)
    if hint is not None:
        start = np.array([hint.get(n, 0.0) for n in model.param_names])
        r1, m1 = _fit(model, start=start, maxiter=5000)
        tried = [('B 계수를 초기값으로(lbfgs, maxiter=5000)', r1, m1)]
        if not _converged(r1):
            rn, _ = _fit(model, start=start, method='nm', maxiter=3000, maxfun=6000)
            r2, m2 = _fit(model, start=rn.params.values, maxiter=5000)
            tried.append(('B 계수 초기값, nm 후 lbfgs(maxiter=5000)', r2, m2))
    else:
        res, msgs = _fit(model)
        tried = [('기본(lbfgs, maxiter=1000)', res, msgs)]
        if retry and not _converged(res):
            r1, m1 = _fit(model, start=res.params.values, maxiter=5000)
            tried.append(('기본 결과에서 이어서(maxiter=5000)', r1, m1))
            if not _converged(r1):
                r2, m2 = _fit(model, start=alt_start(model, ytr, Xtr), maxiter=5000)
                tried.append(('다른 초기값(차분 회귀, maxiter=5000)', r2, m2))
    best = next((t for t in tried if _converged(t[1])), None) or max(tried, key=lambda t: t[1].llf)
    note = best[0] + ('' if _converged(best[1]) else ' [미수렴, loglike 최대 결과 사용]') + ('' if len(tried) == 1 else f' (시도 {len(tried)}회)')
    return best[1], best[2], dropped, X, note

**`validate`**: 계수 고정(refit=False)으로 2024년을 하루씩 앞 예측. 가린 날을 뺀 날의 RMSE, MAPE(원래 단위).

In [10]:
def validate(res, y, X, total, window=None):
    """계수 고정(refit=False)으로 2024년을 하루씩 앞 예측. 가린 날을 뺀 날의 RMSE, MAPE(원래 단위).
    window를 주면 그 날들(특보일+3일) 중 가리지 않은 날만으로도 계산"""
    full = res.apply(y, exog=X, refit=False)                    # 2014~2024 전체에 같은 계수 적용(2024 y는 전날까지만 예측에 쓰임)
    pred = np.exp(full.get_prediction(start=VALID_START, end=VALID_END, dynamic=False).predicted_mean)
    act = total[VALID_START:VALID_END]
    ok = y[VALID_START:VALID_END].notna()                      # 가리기된 날 제외
    err = act[ok] - pred[ok]
    naive = total.shift(7)[VALID_START:VALID_END][ok]            # 참고: 7일 전 같은 요일 값
    out = {'n_days': int(ok.sum()), 'n_masked': int((~ok).sum()),
           'RMSE': float(np.sqrt((err ** 2).mean())), 'MAPE': float((err.abs() / act[ok]).mean() * 100),
           'naive7_RMSE': float(np.sqrt(((act[ok] - naive) ** 2).mean())), 'naive7_MAPE': float(((act[ok] - naive).abs() / act[ok]).mean() * 100)}
    detail = pd.DataFrame({'actual': act, 'pred': pred, 'ok': ok})
    if window is not None:
        m = ok & window
        e = act[m] - pred[m]
        out.update(n_alert=int(m.sum()), RMSE_alert=float(np.sqrt((e ** 2).mean())), MAPE_alert=float((e.abs() / act[m]).mean() * 100))
        detail['window'] = window
    out['detail'] = detail
    return out

**`naive_station`**: 팀원 `Naive_SARIMA.ipynb`를 재현합니다. 이용량(원 단위)에서 코로나 기간만 NaN으로 두고, 역별 차수로 2014~2023에 적합(`enforce_stationarity=False`, `enforce_invertibility=False`, `maxiter=100`)한 뒤 ① 2024년 366일을 한 번에 예측(`get_forecast`)하고 ② 같은 계수로 하루 앞 예측(`apply`)합니다.

In [11]:
def naive_station(d, st):
    s = d[d.station == st].set_index('date').sort_index().asfreq('D')
    y = s.total.copy()
    y[NAIVE_COVID[0]:NAIVE_COVID[1]] = np.nan                           # 팀원 노트북: 코로나 기간만 NaN
    order, seasonal = NAIVE_ORDERS[st]
    fitted = SARIMAX(y[:TRAIN_END], order=order, seasonal_order=seasonal, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=100)
    idx = y[VALID_START:VALID_END].index
    once = fitted.get_forecast(steps=len(idx)).predicted_mean               # 2023-12-31까지의 자료로 2024년 전체를 한 번에
    once.index = idx
    roll = fitted.apply(y, refit=False).get_prediction(start=VALID_START, end=VALID_END, dynamic=False).predicted_mean   # 하루 앞
    return once, roll

**`score`**: 평가일(`ok`)만으로 원 단위 MAE, RMSE, MAPE(%)를 계산합니다.

In [12]:
def score(actual, pred, ok):
    a, p = actual[ok], pred[ok]
    e = a - p
    return {'n_days': int(ok.sum()), 'MAE': float(e.abs().mean()), 'RMSE': float(np.sqrt((e ** 2).mean())), 'MAPE': float((e.abs() / a).mean() * 100)}

**`station_task`**: 한 역에서 5개 모형을 모두 적합해 (가) 하루 앞 롤링과 (나) 365일 한 번에 예측을 만듭니다. 평가일은 우리 마스크(N 모형의 y 관측일)로 정합니다. 우리 모형의 (나)는 계수를 고정한 채 `get_prediction(dynamic=0)`으로 2024-01-01부터 366일을 한 번에 예측합니다 (2023-12-31까지의 자료만 사용, 외생변수는 실제값). 반환: 지표 행, 일별 예측, 적합 요약, 팀원 방식의 366일 전체 MAE.

In [13]:
def station_task(st):
    warnings.filterwarnings('ignore')
    d = load()
    y0, _, grp, total = prepare(d, st, True, 'N')
    ok = y0[VALID_START:VALID_END].notna()                                  # 평가일: 우리 마스크로 가린 날 제외
    act = total[VALID_START:VALID_END]
    rows, dets, fits = [], [], []
    def add(model, mode, pred):
        pred = pred.reindex(act.index)
        rows.append({'역': st, '유형': grp, '모형': model, '방식': mode, **score(act, pred, ok)})
        dets.append(pd.DataFrame({'역': st, '모형': model, '방식': mode, 'date': act.index, 'actual': act.values, 'pred': pred.values, 'ok': ok.values}))
    once, roll = naive_station(d, st)
    add('Naive', '(가) 하루 앞 롤링', roll)
    add('Naive', '(나) 365일 한 번에', once)
    repro = {'역': st, '팀원 검증 MAE': NAIVE_VALID_MAE[st], '재현 검증 MAE': float((act - once.reindex(act.index)).abs().mean())}     # 팀원 방식: 366일 전체
    for m in ['N', 'A', 'B_lag', 'B']:
        y, X, _, tot = prepare(d, st, True, m)
        res, msgs, dropped, Xu, note = fit_station(y, X, ORDER, SEASONAL_ORDER, True)
        v = validate(res, y, Xu, tot)                                       # (가) 하루 앞 롤링 (03번과 같은 함수)
        add(m, '(가) 하루 앞 롤링', v['detail']['pred'])
        full = res.apply(y, exog=Xu, refit=False)
        once_m = np.exp(full.get_prediction(start=VALID_START, end=VALID_END, dynamic=0).predicted_mean)    # (나) 366일 한 번에
        add(m, '(나) 365일 한 번에', once_m)
        fits.append({'역': st, '모형': m, '수렴': bool(res.mle_retvals.get('converged')), '적합': note})
    return rows, pd.concat(dets, ignore_index=True), fits, repro

**9개 역 병렬 실행**: 역마다 5개 모형을 적합합니다 (약 5~10분).

In [14]:
out = Parallel(n_jobs=WORKERS)(delayed(station_task)(st) for st in STATIONS)
res_all = pd.DataFrame([r for o in out for r in o[0]])
pred_all = pd.concat([o[1] for o in out], ignore_index=True)
fits = pd.DataFrame([r for o in out for r in o[2]])
repro = pd.DataFrame([o[3] for o in out])
MODELS = ['Naive', 'N', 'A', 'B_lag', 'B']
MODES = ['(가) 하루 앞 롤링', '(나) 365일 한 번에']
print('지표 행', res_all.shape, '| 일별 예측', pred_all.shape, '| 미수렴 적합:', int((~fits.수렴).sum()), '/', len(fits))

지표 행 (90, 8) | 일별 예측 (32940, 7) | 미수렴 적합: 0 / 36


**팀원 결과 재현 확인**: Naive를 팀원 방식(366일 전체 평가)으로 계산한 2024 검증 MAE가 팀원 노트북이 출력한 값과 같은지 봅니다. 차이가 작으면 Naive 재현이 맞는 것입니다 (statsmodels 버전이 다르면 아주 작은 차이는 있을 수 있음).

In [15]:
repro['차이'] = repro['재현 검증 MAE'] - repro['팀원 검증 MAE']
repro['차이(%)'] = (repro['차이'] / repro['팀원 검증 MAE'] * 100).round(3)
repro.to_csv(RESULTS / 'model_compare_repro.csv', index=False, encoding='utf-8-sig')
repro.round(2)

,역,팀원 검증 MAE,재현 검증 MAE,차이,차이(%)
0,시청,9992.34,9992.71,0.37,0.0
1,역삼,11391.89,11391.89,-0.00,-0.0
2,선릉,9831.97,9831.97,-0.00,-0.0
3,신촌,6030.86,6030.86,0.00,0.0
4,혜화,5324.01,5324.01,0.00,0.0
5,신사,4282.15,4282.15,-0.00,-0.0
6,서울대입구,6575.21,6575.21,0.00,0.0
7,수유,3903.46,3903.46,-0.00,-0.0
8,연신내,3800.23,3800.23,-0.00,-0.0


**03번 결과와 같은 값인지 확인**: 우리 모형 N / A / B_lag / B의 (가) 하루 앞 롤링 MAPE가 03번이 저장한 `sarimax_NABC_compare.csv`의 MAPE와 같아야 합니다.

In [16]:
old = pd.read_csv(RESULTS / 'sarimax_NABC_compare.csv')[['역', '모형', 'MAPE_전체']]
new = res_all[(res_all.방식 == MODES[0]) & res_all.모형.isin(['N', 'A', 'B_lag', 'B'])][['역', '모형', 'MAPE']]
chk = old.merge(new, on=['역', '모형'])
print('03번과 MAPE 최대 차이:', float((chk.MAPE_전체 - chk.MAPE).abs().max()), '(%p) | 비교한 (역, 모형):', len(chk))

03번과 MAPE 최대 차이: 4.440892098500626e-16 (%p) | 비교한 (역, 모형): 36


**역별 MAPE 표**: 유형 순서로 역을 놓고, 방식별로 5개 모형의 MAPE(%)를 나란히 봅니다. 평가일은 모든 모형이 같습니다.

In [17]:
st_order = [s for g in TYPE_ORDER for s in STATIONS if res_all[res_all.역 == s].유형.iloc[0] == g]
res_all['역'] = pd.Categorical(res_all.역, st_order, ordered=True)
mape = res_all.pivot_table(index=['유형', '역'], columns=['방식', '모형'], values='MAPE', observed=True)
mape = mape.reindex(columns=pd.MultiIndex.from_product([MODES, MODELS]))
mape = mape.reindex(pd.MultiIndex.from_tuples([(g, s) for g in TYPE_ORDER for s in st_order if res_all[(res_all.역 == s)].유형.iloc[0] == g]))
mape.round(2)

(가) 하루 앞 롤링                         (나) 365일 한 번에  \
                                 Naive     N     A B_lag     B         Naive   
Employment           시청           7.06  5.21  5.06  5.14  5.01          9.52   
                     역삼           7.01  2.50  2.50  2.52  2.57          7.02   
                     선릉           6.22  2.49  2.45  2.51  2.46          6.05   
Commercial & Leisure 신촌           4.60  3.16  3.16  3.22  3.13          6.10   
                     혜화           4.24  3.66  3.69  3.71  3.71          5.97   
                     신사           5.23  2.68  2.65  2.70  2.61          5.31   
Residential          서울대입구        4.07  2.13  2.07  2.13  2.02          5.24   
                     수유           3.80  2.14  2.09  2.13  1.98          4.04   
                     연신내          4.02  2.37  2.26  2.30  2.01          3.91   

                                                    
                               N     A B_lag     B  
Employment           시청     9.47  8.60  8.55  8.42  
                     역삼     4.18  4.28  4.09  4.25  
                     선릉     3.90  3.77  3.83  3.88  
Commercial & Leisure 신촌     5.34  5.21  5.34  5.00  
                     혜화     6.82  6.35  6.83  6.38  
                     신사     4.34  4.10  4.42  4.15  
Residential          서울대입구  3.61  3.29  3.46  3.20  
                     수유     3.73  3.44  3.64  3.24  
                     연신내    3.28  3.03  3.09  2.73

**유형별 평균**: 역별 지표를 유형(Employment / Commercial & Leisure / Residential)별로 평균합니다 (MAPE, MAE, RMSE). 전체 평균 행도 함께 봅니다.

In [18]:
by_type = res_all.groupby(['방식', '유형', '모형'], observed=True)[['n_days', 'MAE', 'RMSE', 'MAPE']].mean().round(2).reset_index()
tot = res_all.groupby(['방식', '모형'], observed=True)[['n_days', 'MAE', 'RMSE', 'MAPE']].mean().round(2).reset_index()
tot.insert(1, '유형', '전체(9개 역)')
by_type = pd.concat([by_type, tot], ignore_index=True)
by_type['모형'] = pd.Categorical(by_type.모형, MODELS, ordered=True)
by_type = by_type.sort_values(['방식', '유형', '모형']).reset_index(drop=True)
by_type.to_csv(RESULTS / 'model_compare_by_type.csv', index=False, encoding='utf-8-sig')
by_type.pivot_table(index=['방식', '유형'], columns='모형', values='MAPE', observed=True).reindex(columns=MODELS).round(2)

모형                                  Naive     N     A  B_lag     B
방식            유형                                                  
(가) 하루 앞 롤링   Commercial & Leisure   4.69  3.17  3.16   3.21  3.15
              Employment             6.76  3.40  3.34   3.39  3.35
              Residential            3.97  2.21  2.14   2.19  2.00
              전체(9개 역)               5.14  2.93  2.88   2.93  2.83
(나) 365일 한 번에 Commercial & Leisure   5.79  5.50  5.22   5.53  5.18
              Employment             7.53  5.85  5.55   5.49  5.52
              Residential            4.40  3.54  3.25   3.39  3.06
              전체(9개 역)               5.91  4.96  4.68   4.81  4.58

**Naive → N → A / B 단계별 개선**: 유형별 평균 MAPE에서 ① **Naive → N**(마스크·달력 더미·log 모형으로 바꾼 효과), ② **N → A**(특보 추가), ③ **N → B_lag**(전날까지의 날씨 추가), ④ **N → B**(당일 날씨까지 추가)의 개선을 %p와 상대 %로 정리합니다. 양수는 MAPE가 줄어든(좋아진) 것입니다. `results/model_compare_steps.csv`로 저장합니다.

In [19]:
mt = by_type.pivot_table(index=['방식', '유형'], columns='모형', values='MAPE', observed=True)[MODELS]
steps = pd.DataFrame(index=mt.index)
for m in MODELS:
    steps[f'MAPE {m}'] = mt[m]
steps['Naive→N (%p)'] = mt['Naive'] - mt['N']
steps['Naive→N (%)'] = (mt['Naive'] - mt['N']) / mt['Naive'] * 100
for m in ['A', 'B_lag', 'B']:
    steps[f'N→{m} (%p)'] = mt['N'] - mt[m]
    steps[f'N→{m} (%)'] = (mt['N'] - mt[m]) / mt['N'] * 100
steps = steps.round(2)
steps.to_csv(RESULTS / 'model_compare_steps.csv', encoding='utf-8-sig')
steps

MAPE Naive  MAPE N  MAPE A  MAPE B_lag  \
방식            유형                                                             
(가) 하루 앞 롤링   Commercial & Leisure        4.69    3.17    3.16        3.21   
              Employment                  6.76    3.40    3.34        3.39   
              Residential                 3.97    2.21    2.14        2.19   
              전체(9개 역)                    5.14    2.93    2.88        2.93   
(나) 365일 한 번에 Commercial & Leisure        5.79    5.50    5.22        5.53   
              Employment                  7.53    5.85    5.55        5.49   
              Residential                 4.40    3.54    3.25        3.39   
              전체(9개 역)                    5.91    4.96    4.68        4.81   

                                    MAPE B  Naive→N (%p)  Naive→N (%)  \
방식            유형                                                        
(가) 하루 앞 롤링   Commercial & Leisure    3.15          1.52        32.41   
              Employment              3.35          3.36        49.70   
              Residential             2.00          1.76        44.33   
              전체(9개 역)                2.83          2.21        43.00   
(나) 365일 한 번에 Commercial & Leisure    5.18          0.29         5.01   
              Employment              5.52          1.68        22.31   
              Residential             3.06          0.86        19.55   
              전체(9개 역)                4.58          0.95        16.07   

                                    N→A (%p)  N→A (%)  N→B_lag (%p)  \
방식            유형                                                      
(가) 하루 앞 롤링   Commercial & Leisure      0.01     0.32         -0.04   
              Employment                0.06     1.76          0.01   
              Residential               0.07     3.17          0.02   
              전체(9개 역)                  0.05     1.71          0.00   
(나) 365일 한 번에 Commercial & Leisure      0.28     5.09         -0.03   
              Employment                0.30     5.13          0.36   
              Residential               0.29     8.19          0.15   
              전체(9개 역)                  0.28     5.65          0.15   

                                    N→B_lag (%)  N→B (%p)  N→B (%)  
방식            유형                                                    
(가) 하루 앞 롤링   Commercial & Leisure        -1.26      0.02     0.63  
              Employment                   0.29      0.05     1.47  
              Residential                  0.90      0.21     9.50  
              전체(9개 역)                     0.00      0.10     3.41  
(나) 365일 한 번에 Commercial & Leisure        -0.55      0.32     5.82  
              Employment                   6.15      0.33     5.64  
              Residential                  4.24      0.48    13.56  
              전체(9개 역)                     3.02      0.38     7.66

**역별 단계별 개선 (참고)**: 같은 비교를 역마다 봅니다. N 대비 A, B_lag, B가 MAPE를 줄인 역이 몇 개인지도 셉니다.

In [20]:
ms = res_all.pivot_table(index=['방식', '역'], columns='모형', values='MAPE', observed=True)[MODELS]
cnt = pd.DataFrame({f'N보다 좋아진 역 수 ({m})': (ms['N'] - ms[m] > 0).groupby(level='방식').sum() for m in ['A', 'B_lag', 'B']})
cnt['Naive보다 N이 좋아진 역 수'] = (ms['Naive'] - ms['N'] > 0).groupby(level='방식').sum()
cnt

,N보다 좋아진 역 수 (A),N보다 좋아진 역 수 (B_lag),N보다 좋아진 역 수 (B),Naive보다 N이 좋아진 역 수
방식,,,,
(가) 하루 앞 롤링,8,4,7,9
(나) 365일 한 번에,8,6,8,8


**저장**: 역 × 모형 × 방식 지표, 일별 예측, 적합 요약.

In [21]:
res_all['역'] = res_all.역.astype(str)
res_all.to_csv(RESULTS / 'model_compare_by_station.csv', index=False, encoding='utf-8-sig')
pred_all.to_csv(RESULTS / 'model_compare_pred2024.csv', index=False, encoding='utf-8-sig')
fits.to_csv(RESULTS / 'model_compare_fits.csv', index=False, encoding='utf-8-sig')
print('저장:', res_all.shape, pred_all.shape, fits.shape)

저장: (90, 8) (32940, 7) (36, 4)
